In [19]:
!pip install -q delta-spark pyspark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.8 MB/s eta 0:00:00


In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Parquet_format").getOrCreate()
print("spark session ready")

spark session ready


In [ ]:
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000),
(103,"Rahul Sharma","Mumbai","Dermatology",1500),
(104,"Priya Nair","Bangalore","Cardiology",5000),
(105,"Vikram Singh","Chennai","Neurology",7000)
]
columns = ["visit_id","patient_name","city","department","consultation_fee"]
df = spark.createDataFrame(data, columns)
df.show()

In [5]:
df.write.mode("overwrite").parquet("/tmp/patient_parquet")

In [ ]:
parquet_df = spark.read.parquet("/tmp/patient_parquet")
parquet_df.show()

In [ ]:
parquet_df.printSchema()

In [ ]:
parquet_df.select("patient_name", "city").show()

In [ ]:
from pyspark.sql.functions import col
parquet_df.filter(col("consultation_fee") > 3000).show()

In [10]:
df.write.mode("overwrite").partitionBy("city")\
  .parquet("/tmp/patient_parquet_partitioned")

In [ ]:
partition_parquet_df= spark.read.parquet("/tmp/patient_parquet_partitioned")
partition_parquet_df.show()

In [ ]:
partition_parquet_df.filter(col("city") == "Hyderabad").show()

In [14]:
new_data = [
(106,"Ananya Das","Kolkata","Orthopedics",3000)
]
new_df = spark.createDataFrame(new_data, columns)
new_df.write.mode("append").parquet("/tmp/patient_parquet")

In [15]:
df.write.mode("overwrite").parquet("/tmp/patient_parquet")

In [17]:
spark.sql(""" CREATE TABLE IF NOT EXISTS patient_parquet_table
USING PARQUET
LOCATION '/tmp/patient_parquet';  """)

DataFrame[]

In [ ]:
spark.sql(" select * from patient_parquet_table").show()